# First-place clean-room behavioral reproduction
Run cells in order. The program check preserves the full model, 400x345 grid, batch 16, BN and BF16; it only reduces wells and epochs.

In [ ]:
from pathlib import Path
import os, subprocess, sys

code_candidates = [
    Path.cwd(),
    Path('/root/autodl-tmp/rogii-reproduction'),
    Path('/root/autodl-tmp'),
]
CODE_ROOT = next((p for p in code_candidates if (p / 'first_place_cleanroom').is_dir()), None)
data_candidates = [
    Path('/root/autodl-tmp/rogii-wellbore-geology-prediction'),
    CODE_ROOT if CODE_ROOT else Path('.'),
]
DATA_ROOT = next((p for p in data_candidates if (p / 'train').is_dir()), None)
WORK_ROOT = Path('/root/autodl-tmp/first_place_cleanroom')
assert CODE_ROOT is not None, 'Upload first_place_cleanroom/ under /root/autodl-tmp or the current directory'
assert DATA_ROOT is not None, 'Competition train/ directory was not found'
os.chdir(CODE_ROOT)
print({'code': str(CODE_ROOT), 'data': str(DATA_ROOT), 'work': str(WORK_ROOT),
       'python': sys.executable})

def run_module(module, *arguments):
    command = [sys.executable, '-u', '-m', module, *map(str, arguments)]
    print('RUN:', ' '.join(command), flush=True)
    process = subprocess.Popen(command, cwd=CODE_ROOT, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise RuntimeError({'returncode': code, 'command': command})


In [ ]:
# Install only declared runtime dependencies.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                str(CODE_ROOT / 'first_place_cleanroom/requirements.txt')], check=True)

In [ ]:
# Mandatory disclosed-contract, simulation-distribution and representation gates.
run_module('first_place_cleanroom.contract_audit', '--data-root', DATA_ROOT, '--model-forward')
run_module('first_place_cleanroom.simulation_audit', '--data-root', DATA_ROOT, '--wells', '128')
run_module('first_place_cleanroom.audit', '--data-root', DATA_ROOT,
           '--version', '0719_V1', '--model-forward')

In [ ]:
# Build complete fold-clean PF and XY caches. This is resumable through per-well NPZ files.
run_module('first_place_cleanroom.prepare',
           '--data-root', DATA_ROOT,
           '--output-root', WORK_ROOT / 'cache',
           '--component', 'all', '--split', 'both', '--workers', '8')
run_module('first_place_cleanroom.contract_audit', '--data-root', DATA_ROOT,
           '--cache-root', WORK_ROOT / 'cache')

In [ ]:
# Unchanged-semantics program check: full 0803 model, batch 16, BF16, PF and XY.
args = [
    '--data-root', DATA_ROOT, '--cache-root', WORK_ROOT / 'cache',
    '--output-root', WORK_ROOT / 'program_check',
    '--version', '0803_V2', '--fold', '1', '--seed', '0',
    '--epochs', '2', '--limit-train', '32', '--limit-valid', '16',
]
run_module('first_place_cleanroom.train', *args)

In [ ]:
# First comparable gate: one complete 0719 Fold 1 / seed 0 member, 300 epochs.
args = [
    '--data-root', DATA_ROOT, '--cache-root', WORK_ROOT / 'cache',
    '--output-root', WORK_ROOT / 'models',
    '--version', '0719_V1', '--fold', '1', '--seed', '0',
]
run_module('first_place_cleanroom.train', *args)

In [ ]:
# Run only after the full Fold-1 metric passes the discrepancy gate.
run_module('first_place_cleanroom.run_all',
           '--stage', 'train', '--data-root', DATA_ROOT, '--work-root', WORK_ROOT)

In [ ]:
# Honest OOF after all 105 members exist. This must precede any reproduction claim.
run_module('first_place_cleanroom.run_all',
           '--stage', 'oof', '--data-root', DATA_ROOT, '--work-root', WORK_ROOT)

In [ ]:
# Optional final inference, only after all 105 best.pt checkpoints exist.
run_module('first_place_cleanroom.run_all',
           '--stage', 'infer', '--data-root', DATA_ROOT, '--work-root', WORK_ROOT,
           '--submission', WORK_ROOT / 'submission.csv')